# `pyspark`: Exploratory Data Analysis

* Spark is a unified data processing framework allowing pre-processing, ETL, machine learning, streaming, and graph analysis
* Spark SQL is a set of data structures and extensions that allow you to consume data from a wide variety of sources: RDB, file, database, and other sources
* The primary structure for working with structured data is the `DataFrame`
* Use-Case: Analyze US flight data
  - Shows an example of how to work semi-structured data (JSON)

_The examples and case study in this notebook has been adapted from Chapter 2: Predicting Flight Delays using Apache Spark Machine Learning of "Getting Started with Apache Spark" by Carol McDonald. The original examples were developed in Scala._

#### Environment Notes
This notebook includes setup code and options for working with data stored in MinIO (S3) using `pyspark`.

### Spark and Storage Configuration

In [ ]:
import os, sys, posixpath

# User Options
USER_NAMESPACE = os.environ.get('HOSTNAME')

# Spark Options
SPARK_DRIVER_RAM = '3g'
SPARK_DRIVER_MAXRESULTSIZE = '4096m'
SPARK_WORKER_RAM = '6g'

# Access credentials for Object Storage
OBJECT_STORAGE_URL = os.environ.get('OBJECTS_ENDPOINT')
OBJECT_STORAGE_ACCESSID = os.environ.get('OBJECTS_ACCESSID')
OBJECT_STORAGE_SECRET = os.environ.get('OBJECTS_SECRET')
OBJECTS_PLAYGROUND_BUCKET = 'playground'
OBJECTS_USER_PREFIX = USER_NAMESPACE
OBJECT_CONTAINER = 'web-age'
STORAGEURL_PLAYGROUND = 'playground'

# Spark Runtime Options
os.environ['PYSPARK_PYTHON'] = 'python3'
os.environ['PYSPARK_DRIVER_PYTHON'] = 'python3'

### Configure Spark for Local Context and Initialize `SparkContext` (Application Driver)

In [ ]:
# Manage PySpark Runtime Options
import os

PACKAGE_OPTIONS = '--packages %s ' % ','.join((        
        # 'org.apache.spark:spark-avro_2.12:2.4.4',
    ))

JAR_OPTIONS = '--jars %s ' % ','.join((
        '/opt/spark/jars/joda-time-2.9.9.jar',
        '/opt/spark/jars/httpclient-4.5.3.jar',
        '/opt/spark/jars/aws-java-sdk-s3-1.11.534.jar',
        '/opt/spark/jars/aws-java-sdk-kms-1.11.534.jar',
        '/opt/spark/jars/aws-java-sdk-dynamodb-1.11.534.jar',
        '/opt/spark/jars/aws-java-sdk-core-1.11.534.jar',
        '/opt/spark/jars/aws-java-sdk-1.11.534.jar',
        '/opt/spark/jars/hadoop-aws-3.1.2.jar',
        '/opt/spark/jars/slf4j-api-1.7.29.jar',
        '/opt/spark/jars/slf4j-log4j12-1.7.29.jar',
    ))

os.environ['PYSPARK_SUBMIT_ARGS'] = JAR_OPTIONS + ' pyspark-shell'
os.environ.get('PYSPARK_SUBMIT_ARGS')

* `SparkSession` objects help to coordinate the components of a Spark program
  - When using the Spark shell, `SparkSession` objects are created automatically
  - For other programs (`spark-submit` and notebooks), they need to be created manually
* Spark running in cluster mode needs to communicate with a master, that manages the setup of drivers within a cluster computing system
  - Supported masters include Kubernetes, Mesosphere, and Hadoop
  - Example code below shows the initialization of a local master.

In [ ]:
import pyspark

conf = pyspark.SparkConf()

# Memory
conf.set('spark.driver.memory', SPARK_DRIVER_RAM)
conf.set("spark.executor.memory", SPARK_WORKER_RAM)
conf.set('spark.driver.maxResultSize', SPARK_DRIVER_MAXRESULTSIZE)

# The DNS alias for the Spark driver. Required by executors to report status.
conf.set("spark.driver.host", os.environ.get('HOSTNAME'))

# Port which the Spark shell should bind to and to which executors will report progress
conf.set("spark.driver.port", "20020")

# Configure S3 Object Storage as filesystem, pass MinIO credentials
conf.set("spark.hadoop.fs.s3a.endpoint", OBJECT_STORAGE_URL) \
    .set("spark.hadoop.fs.s3a.access.key", OBJECT_STORAGE_ACCESSID) \
    .set("spark.hadoop.fs.s3a.secret.key", OBJECT_STORAGE_SECRET) \
    .set("spark.hadoop.fs.s3a.fast.upload", True) \
    .set("spark.hadoop.fs.s3a.path.style.access", True) \
    .set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")

# Configure Avro options
conf.set('avro.mapred.ignore.inputs.without.extension', 'true')

# Initialize spark context, create executors
conf.setAppName('local-arcos-extract-instructor')
sc = pyspark.SparkContext(conf=conf)

# Initialize Spark Session
from pyspark.sql import SparkSession
spark = SparkSession(sc)

Create a small distributed job to test the connection. _It's always a good idea to ensure that the context works and that worker nodes are able to communicate with the master._

In [ ]:
# Create a distributed data set to test to the session
t = sc.parallelize(range(10))

# Calculate the approximate sum of values in the dataset
r = t.sumApprox(3)
print('Approximate sum: %s' % r)

#### Import Visualization Tools

In [ ]:
import matplotlib
import seaborn as sns

%matplotlib inline

## U.S Flight Data
Data used in this notebook is available from the [United States Department of Transportation (DOT)](https://www.transtats.bts.gov/DL_SelectFields.asp?Table_ID=236&DB_Short_Name=On-Time). It has been deployed as JSON objects within MinIO (a storage solution that implements an interface similar to the one provided by Amazon's Simple Storage Solution - S3). _JSON is an example of a semi-structured form of data. It uses key/value paris to encode information and [includes support](https://www.json.org/json-en.html) for primitive types such as strings, arrays, numbers, booleans, and arrays._

The flight data includes the following fields:

* id: ID composed of carrier, date, origin, destination, flight number
* dofW: day of week (1=Monday, 7=Sunday)
* carrier: carrier code
* origin: origin airport code
* dest: destination airport code
* crsdephour: scheduled departure hour
* crsdeptime: scheduled departure time
* depdelay: departure delay in minutes
* crsarrtime: scheduled arrival time
* arrdelay: arrival delay minutes
* crselapsedtime: elapsed time
* dist: distance

Example:

```json
{
“_id”: “AA_2017-01-01_ATL_LGA_1678”,
“dofW”: 7,
“carrier”: “AA”,
“src”: “ATL”,
“dst”: “LGA”,
“crsdephour”: 17,
“crsdeptime”: 1700,
“depdelay”: 0.0,
“crsarrtime”: 1912,
“arrdelay”: 0.0,
“crselapsedtime”: 132.0,
“dist”: 762.0
}
```

## `RDD`, `DataFrame`, and `DataSet`
* Resilient Distributed Dataset (RDD)
  - Fundamental abstraction in Spark
  - Represents data across machines in a cluster
  - Data is "partioned" across the program's workers
  - Low level structure not usually used in favor of the `DataSet` and `DataFrame`

* `DataSet` is a distributed colleciton of typed objects
  - Partioned across workers (superset of functionality on top of RDD)
  - Dataset objects are manipulated using map, flatMap, filter
  - [Dataset API is not available in Python](https://spark.apache.org/docs/latest/sql-programming-guide.html)
  - _Many of the benefits of the Dataset API are available from Python due to the dynamic nature of the language._

* `DataFrame` is a `Dataset` with named columns
  - Supported in Scala, Java, Python and R
  - Primay interface of Spark SQL
  - Implements an optimized interface due to type annotations

In [ ]:
# When loading data it is possible to allow Spark to infer the data structure,
# or the data structure can be imposed. Imposing a structure often results 
# in a cleaner load
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# Spark sql.functions provides a complete library of methods that can be used
# for transforming, aggregating, and managing the structure of the data
import pyspark.sql.functions as sparkfunc

# Create alias to DataFreameStatFunctions
from pyspark.sql import DataFrameStatFunctions as statfunc

In [ ]:
# Create schema for the flight data: StructField arguments
# "name": name of the data column
# "dataType": type of data
# "nullable": indicates if values of the field can have null values
flights_schema = StructType([
    StructField('id', StringType(), True),
    StructField('month', IntegerType(), True),
    StructField('dofW', IntegerType(), True),
    StructField('carrier', StringType(), True),
    StructField('src', StringType(), True),
    StructField('dst', StringType(), True),
    StructField('crsdephour', IntegerType(), True),
    StructField('crsdeptime', DoubleType(), True),
    StructField('crsarrtime', DoubleType(), True),
    StructField('crselapsedtime', DoubleType(), True),
    StructField('depdelay', DoubleType(), True),
    StructField('dist', DoubleType(), True),
    
    # Fields used later for machine learning
    StructField('label', DoubleType(), True),
    StructField('pred_dtree', DoubleType(), True),
])

### Load and Inspect the Structure of Data

In [ ]:
# Load data from JSON and impose data
flights_df = spark.read.format('json') \
    .option('inferSchema', False) \
    .schema(flights_schema) \
    .load('s3a://web-age/wa2843/dot.flights.2018.json')

# Inspect data structure
flights_df.printSchema()

In [ ]:
# PySpark DataFrames include a named API that is similar to that of Pandas
flights_df.dtypes

Commonly used dataframe actions:

* `show(n)`: Display the first n rows in a tabular form
* `take(n)`: Return the first n objects in a DataFrame in an array
  - Pulls the data from executors into the driver
* `count`: Count the number of rows available in the DataFrame

In [ ]:
# Show the top rows of the data frame to inspect results of import
flights_df.show()

### DataFrame Transformations

* `join`: Combine one DataFrame with another using a join expression
* `groupBy`: Group the data frame by the provided column for aggregations or other transformations

In [ ]:
# Explore the number of flights by the carrier
flights_df.groupBy('carrier', 'month').count().orderBy('month').show()

In [ ]:
# Calculate descriptive statistics on the columns. Spark uses SQL terms to describe
# its data transforms. One of these is `select` which allows for choosing a subset of data
# for display
flights_df.select('dofW', 'crsdephour', 'crsdeptime', 'crsarrtime', 'crselapsedtime', 'depdelay') \
    .describe().show()

#### [Caching Data in Memory](https://spark.apache.org/docs/latest/sql-performance-tuning.html)
Spark SQL can cache tables using an in-memory columnar format by calling `spark.cata.log.cacheTable('tableName')` or `dataFrame.cache()`.

* Spark SQL will scan only required columns and will automatically tune compression to minimize memory usage and GC pressure.
* `spark.catalog.uncacheTable('tableName')` removes the table from memory

In [ ]:
# Data can be cahced in memory to increase the performance of subsequent operations
flights_df.cache()

# Create Table view of DataFrame for Spark SQL
flights_df.createOrReplaceTempView('flights')

# Cache flights tabe in column format in memory
spark.catalog.cacheTable('flights')

In [ ]:
# Make use of a dataframe transformation to find flights delayed more than 40 minutes,
# and then order the results by the delay in reverse order
flights_df.filter(sparkfunc.col('depdelay') > 40) \
    .orderBy(sparkfunc.desc('depdelay')) \
    .select('carrier', 'src', 'dst', 'depdelay', 'crsdephour') \
    .show(5)

#### Data Aggregation and Visualization

* `groupBy` allows for data to be grouped by a variable of interest and aggregated
* Aggregations summarize data by metrics of interest such as mean, median, count, etc.
  - Provide the functional foundation for `describe`
* Useful to visualize data. Because of the size of the underlying data, it is often a good idea to sample or aggregate before visualization.
* When working in a Python/Jupyter environment, visualization tools such as Seaborn are often used

##### Delays By Carrier

In [ ]:
# Which carrier has the most delayed departures
carrier_delay_pdf = flights_df.filter(sparkfunc.col('depdelay') > 40) \
    .groupBy('carrier') \
    .agg(
        sparkfunc.count('carrier').alias('delay_count'),
        sparkfunc.min('depdelay').alias('delay_min'),
        sparkfunc.mean('depdelay').alias('delay_avg'),
        sparkfunc.max('depdelay').alias('delay_max')).toPandas()

In [ ]:
# Visualize the count of departure delays by carrier
ax = sns.barplot(x='carrier', y='delay_count', data=carrier_delay_pdf)
ax.set(ylabel='Number of Delayed Flights', xlabel='Carrier')

##### Delays By Day of Week

In [ ]:
# Is there a day of the week where delays are more likely
week_delay_pdf = flights_df.filter(sparkfunc.col('depdelay') > 40) \
    .groupBy('dofW') \
    .agg(
        sparkfunc.count('carrier').alias('delay_count'),
        sparkfunc.min('depdelay').alias('delay_min'),
        sparkfunc.mean('depdelay').alias('delay_avg'),
        sparkfunc.max('depdelay').alias('delay_max')).toPandas()

In [ ]:
# Visualize the count of departure delays by day of the week
ax = sns.barplot(x='dofW', y='delay_count', data=week_delay_pdf)
ax.set(ylabel='Number of Delayed Flights', xlabel='Day of the Week')

##### Delay By Time of Day
Is there a time of the day when delays are more likely?

In [ ]:
# Is there an hour of the day where delays are more likely
hour_delay_pdf = flights_df.filter(sparkfunc.col('depdelay') > 40) \
    .groupBy('crsdephour') \
    .agg(
        sparkfunc.count('carrier').alias('delay_count'),
        sparkfunc.min('depdelay').alias('delay_min'),
        sparkfunc.mean('depdelay').alias('delay_avg'),
        sparkfunc.max('depdelay').alias('delay_max')).toPandas()

ax = sns.barplot(x='crsdephour', y='delay_count', data=hour_delay_pdf)
ax.set(ylabel='Number of Delayed Flights', xlabel='Hour of the Day')

##### Delays By Originating Airport
Does the airport from which the flight departs make a difference?

In [ ]:
# Is there a day of the week where departures are more likely
origairport_delay_pdf = flights_df.filter(sparkfunc.col('depdelay') > 40) \
    .groupBy('src') \
    .agg(
        sparkfunc.count('carrier').alias('delay_count'),
        sparkfunc.min('depdelay').alias('delay_min'),
        sparkfunc.mean('depdelay').alias('delay_avg'),
        sparkfunc.max('depdelay').alias('delay_max')) \
    .orderBy(sparkfunc.desc('delay_count')) \
    .toPandas()

ax = sns.barplot(x='src', y='delay_count', data=origairport_delay_pdf)
ax.set(ylabel='Number of Delayed Flights', xlabel='Originating Airport')

##### Trends By Route
Spark can aggregate data by more than a single column, which allows for the analysis of more complex trends. For example, are there specific routes that are more likely to experience delays than others?

* Bar Chart: In an analysis of route, it may be a good idea to narrow the number of airports so that the results can be better inspected.
* Bar Chart: The example code looks at three originating airpots: ORD, LAX, ATL

In [ ]:
route_delay_pdf1 = flights_df \
    .filter((
        sparkfunc.col('src') == 'ORD') 
        | (sparkfunc.col('src') == 'LAX') 
        | (sparkfunc.col('src') == 'ATL')) \
    .filter(sparkfunc.col('depdelay') > 40) \
    .groupBy('src', 'dst') \
    .agg(
        sparkfunc.count('carrier').alias('delay_count'),
        sparkfunc.min('depdelay').alias('delay_min'),
        sparkfunc.mean('depdelay').alias('delay_avg'),
        sparkfunc.max('depdelay').alias('delay_max')) \
    .orderBy(sparkfunc.desc('delay_count')) \
    .toPandas()

ax = sns.catplot(x='src', y='delay_count', hue='dst', kind='bar', data=route_delay_pdf1, 
                height=10, palette='muted')
ax.despine(left=True)
ax.set_ylabels('Number of Delayed Flights')

Standard visualizations (like a grouped bar chart) doesn't always do a good job of making comparisons if there are a large number of categories or values. Heatmaps, on the other hand, can.

In [ ]:
route_delay_pdf2 = flights_df \
    .filter(sparkfunc.col('depdelay') > 40) \
    .groupBy('src', 'dst') \
    .agg(
        sparkfunc.count('carrier').alias('delay_count'),
        sparkfunc.min('depdelay').alias('delay_min'),
        sparkfunc.mean('depdelay').alias('delay_avg'),
        sparkfunc.max('depdelay').alias('delay_max')) \
    .orderBy(sparkfunc.desc('delay_count')) \
    .toPandas()

# Create heatmap (drop null values)
ax = sns.heatmap(route_delay_pdf2.pivot('src', 'dst', 'delay_count'))